# Conservative Patient vs Control Classifier

This Colab notebook uses the labels from the first sheet, `measurement_inform`, as the source of truth.

It avoids reporting the highest tuned result. The default model is locked in advance:

- session-level feature matrix
- leave-one-subject-out evaluation by `patient_code`
- non-distributional metrics only
- median imputation
- standard scaling
- SelectKBest with 20 features
- class-balanced L2 logistic regression with `C=0.3`
- average held-out session probabilities into one prediction per person

Expected conservative performance on this workbook is around **80% balanced accuracy**, but with only 10 individuals the uncertainty is wide. The notebook reports Wilson 95% confidence intervals.

In [ ]:
from pathlib import Path
import json, re, warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
WORKBOOK_PATH = Path('/content/fixations-lisseyejous001-010 (1).xlsx')
if not WORKBOOK_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            WORKBOOK_PATH = Path(next(iter(uploaded.keys())))
    except Exception:
        pass
print('Using workbook:', WORKBOOK_PATH)
assert WORKBOOK_PATH.exists(), 'Upload the Excel workbook or set WORKBOOK_PATH.'

Saving fixations-lisseyejous001-010 (1).xlsx to fixations-lisseyejous001-010 (1).xlsx
Using workbook: fixations-lisseyejous001-010 (1).xlsx


In [ ]:
TARGET_COL = 'GT'
ID_COL = 'patient_code'
PATH_COL = 'path'
POSITIVE_LABEL = 'Patient'

METRIC_SHEETS = [
    'non_distributional_parameters',
    'distributional_aggregated',
    'saccade_non_aggregated',
    'pso_non_aggregated',
    'saccade_with_pso_non_aggregated',
    'drift_non_aggregated',
    'psychopy_non_aggregated',
]
NON_FEATURE_COLS = {'parameter', ID_COL, PATH_COL, 'Unnamed: 0'}


def clean_token(value):
    if pd.isna(value):
        return ''
    text = re.sub(r'[^0-9A-Za-z]+', '_', str(value).strip())
    return re.sub(r'_+', '_', text).strip('_')


def make_metric_column_names(raw, sheet_name):
    axis_row = raw.iloc[0] if len(raw) else pd.Series(dtype=object)
    unit_row = raw.iloc[1] if len(raw) > 1 else pd.Series(dtype=object)
    names, seen, current_metric = [], {}, ''
    for col in raw.columns:
        col_text = str(col)
        if not col_text.startswith('Unnamed'):
            current_metric = clean_token(col_text) or current_metric
        if col in [ID_COL, PATH_COL, 'parameter', 'Unnamed: 0']:
            base = clean_token(col_text)
        else:
            axis = clean_token(axis_row.get(col, ''))
            unit = clean_token(unit_row.get(col, ''))
            parts = [sheet_name, current_metric]
            if axis: parts.append(axis)
            if unit: parts.append(unit)
            base = '__'.join(parts)
        count = seen.get(base, 0)
        seen[base] = count + 1
        names.append(base if count == 0 else f'{base}__{count + 1}')
    return names


def coerce_numeric_frame(df):
    cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    numeric = df[cols].apply(pd.to_numeric, errors='coerce')
    numeric = numeric.loc[:, numeric.notna().any(axis=0)]
    return numeric.replace([np.inf, -np.inf], np.nan)


def iqr(series):
    values = pd.to_numeric(series, errors='coerce').dropna()
    return np.nan if values.empty else float(values.quantile(0.75) - values.quantile(0.25))


def load_metric_sheet(workbook, sheet_name):
    raw = pd.read_excel(workbook, sheet_name=sheet_name)
    raw.columns = make_metric_column_names(raw, sheet_name)
    return raw.iloc[3:].reset_index(drop=True).dropna(subset=[PATH_COL])

In [ ]:
def aggregate_session_level(df, prefix):
    numeric = coerce_numeric_frame(df)
    frame = pd.concat([df[[ID_COL, PATH_COL]], numeric], axis=1).dropna(subset=[PATH_COL])
    grouped = frame.groupby([ID_COL, PATH_COL], dropna=False)
    agg = grouped[numeric.columns].agg(['mean', 'std', 'median', 'min', 'max', iqr])
    agg.columns = [f'{prefix}__{col}__{stat}' for col, stat in agg.columns]
    return agg.reset_index()


def build_session_feature_matrix(workbook):
    # First sheet labels: this is the single source of truth for patient/control.
    measurement = pd.read_excel(workbook, sheet_name='measurement_inform')
    measurement = measurement.dropna(subset=[ID_COL, PATH_COL, TARGET_COL]).copy()
    measurement['target'] = (measurement[TARGET_COL] == POSITIVE_LABEL).astype(int)

    conflicts = measurement.groupby(ID_COL)['target'].nunique()
    if (conflicts > 1).any():
        raise ValueError('Conflicting first-sheet labels for: ' + ', '.join(conflicts[conflicts > 1].index))

    base_cols = [ID_COL, PATH_COL, 'age', 'sex', 'which_eye', 'device_frequency']
    features = measurement[[c for c in base_cols if c in measurement.columns]].copy()
    features = pd.get_dummies(features, columns=[c for c in ['sex', 'which_eye'] if c in features.columns], dummy_na=True)

    # Conservative default: only non-distributional metrics. This avoids thousands of event-derived features.
    nd = load_metric_sheet(workbook, 'non_distributional_parameters')
    nd_features = aggregate_session_level(nd, 'non_distributional_parameters')
    features = features.merge(nd_features, on=[ID_COL, PATH_COL], how='outer')

    labels = measurement[[ID_COL, PATH_COL, 'target']].drop_duplicates([ID_COL, PATH_COL])
    out = labels.merge(features, on=[ID_COL, PATH_COL], how='left')
    X = out.drop(columns=[ID_COL, PATH_COL, 'target']).apply(pd.to_numeric, errors='coerce')
    X = X.loc[:, X.notna().any(axis=0)]
    y = out['target'].astype(int)
    groups = out[ID_COL].astype(str)
    return X, y, groups, out[[ID_COL, PATH_COL, 'target']]


X_session, y_session, groups_session, session_index = build_session_feature_matrix(WORKBOOK_PATH)
print('Sessions:', len(X_session))
print('Individuals:', groups_session.nunique())
print('Features:', X_session.shape[1])
print('Labels are from first sheet: measurement_inform.GT')
display(session_index.groupby([ID_COL, 'target']).size().rename('sessions').reset_index())

Sessions: 59
Individuals: 10
Features: 172
Labels are from first sheet: measurement_inform.GT


,patient_code,target,sessions
0,Jonnal_0001,0,7
1,LissEyeJous004,1,7
2,LissEyeJous005,0,7
3,LissEyeJous006,1,7
4,LissEyeJous007,0,5
5,LissEyeJous_001,0,7
6,LissEyeJous_002,0,7
7,LissEyeJous_010,1,5
8,LissEyejous_008,1,3
9,lisseyejous_009,1,4


In [ ]:
def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    phat = successes / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = z * np.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n) / denom
    return (float(center - half), float(center + half))


def make_locked_model():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('select', SelectKBest(score_func=f_classif, k=20)),
        ('classifier', LogisticRegression(C=0.3, class_weight='balanced', solver='liblinear', max_iter=5000, random_state=RANDOM_STATE)),
    ])


def evaluate_leave_one_subject_out(X, y, groups):
    rows = []
    logo = LeaveOneGroupOut()
    for train_idx, test_idx in logo.split(X, y, groups):
        subject = groups.iloc[test_idx].iloc[0]
        model = make_locked_model()
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        session_prob = model.predict_proba(X.iloc[test_idx])[:, 1]
        subject_prob = float(np.mean(session_prob))
        rows.append({
            ID_COL: subject,
            'actual': int(y.iloc[test_idx].iloc[0]),
            'predicted': int(subject_prob >= 0.5),
            'patient_probability': subject_prob,
            'heldout_sessions': int(len(test_idx)),
        })
    pred = pd.DataFrame(rows)
    y_true, y_pred, y_prob = pred['actual'], pred['predicted'], pred['patient_probability']
    correct = int((y_true == y_pred).sum())
    patient_mask = y_true == 1
    control_mask = y_true == 0
    metrics = {
        'recommended_percentage': f'{balanced_accuracy_score(y_true, y_pred) * 100:.0f}% balanced accuracy',
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'accuracy_95ci_wilson': wilson_ci(correct, len(pred)),
        'patient_recall_95ci_wilson': wilson_ci(int(((y_pred == 1) & patient_mask).sum()), int(patient_mask.sum())),
        'control_recall_95ci_wilson': wilson_ci(int(((y_pred == 0) & control_mask).sum()), int(control_mask.sum())),
        'confusion_matrix_labels': ['Control', 'Patient'],
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
        'confidence': 'Low to moderate at best: only 10 individuals. Treat as exploratory until tested on new subjects.',
    }
    return metrics, pred


metrics, predictions = evaluate_leave_one_subject_out(X_session, y_session, groups_session)
predictions['actual_label'] = predictions['actual'].map({0: 'Control', 1: 'Patient'})
predictions['predicted_label'] = predictions['predicted'].map({0: 'Control', 1: 'Patient'})

print(json.dumps(metrics, indent=2))
print('\nReport this as:', metrics['recommended_percentage'])
print('95% CI for raw accuracy:', metrics['accuracy_95ci_wilson'])
print('Confidence:', metrics['confidence'])
print('\nClassification report:')
print(classification_report(predictions['actual'], predictions['predicted'], target_names=['Control', 'Patient'], zero_division=0))
display(predictions[[ID_COL, 'actual_label', 'predicted_label', 'patient_probability', 'heldout_sessions']])

{
  "recommended_percentage": "80% balanced accuracy",
  "accuracy": 0.8,
  "balanced_accuracy": 0.8,
  "roc_auc": 0.9600000000000001,
  "accuracy_95ci_wilson": [
    0.49015684672072335,
    0.9433190520193067
  ],
  "patient_recall_95ci_wilson": [
    0.3755282641185388,
    0.9637768390302125
  ],
  "control_recall_95ci_wilson": [
    0.3755282641185388,
    0.9637768390302125
  ],
  "confusion_matrix_labels": [
    "Control",
    "Patient"
  ],
  "confusion_matrix": [
    [
      4,
      1
    ],
    [
      1,
      4
    ]
  ],
  "confidence": "Low to moderate at best: only 10 individuals. Treat as exploratory until tested on new subjects."
}

Report this as: 80% balanced accuracy
95% CI for raw accuracy: (0.49015684672072335, 0.9433190520193067)
Confidence: Low to moderate at best: only 10 individuals. Treat as exploratory until tested on new subjects.

Classification report:
              precision    recall  f1-score   support

     Control       0.80      0.80      0.80     

,patient_code,actual_label,predicted_label,patient_probability,heldout_sessions
0,Jonnal_0001,Control,Patient,0.519535,7
1,LissEyeJous004,Patient,Patient,0.969387,7
2,LissEyeJous005,Control,Control,0.172220,7
3,LissEyeJous006,Patient,Patient,0.801206,7
4,LissEyeJous007,Control,Control,0.217068,5
5,LissEyeJous_001,Control,Control,0.202593,7
6,LissEyeJous_002,Control,Control,0.004943,7
7,LissEyeJous_010,Patient,Patient,0.905779,5
8,LissEyejous_008,Patient,Control,0.274389,3
9,lisseyejous_009,Patient,Patient,0.813722,4


In [ ]:
OUTDIR = Path('/content/patient_classifier_outputs')
OUTDIR.mkdir(parents=True, exist_ok=True)

final_model = make_locked_model()
final_model.fit(X_session, y_session)

joblib.dump(
    {
        'model': final_model,
        'feature_columns': list(X_session.columns),
        'labels_source': 'measurement_inform.GT',
        'evaluation': metrics,
        'note': 'Conservative exploratory model. Needs validation on new subjects.',
    },
    OUTDIR / 'conservative_patient_classifier.joblib',
)
predictions.to_csv(OUTDIR / 'leave_one_subject_predictions.csv', index=False)
X_session.assign(target=y_session, patient_code=groups_session.values).to_csv(OUTDIR / 'session_feature_matrix.csv', index=False)
(OUTDIR / 'metrics_summary.json').write_text(json.dumps(metrics, indent=2))

print('Saved files:')
for path in sorted(OUTDIR.iterdir()):
    print('-', path)

Saved files:
- /content/patient_classifier_outputs/conservative_patient_classifier.joblib
- /content/patient_classifier_outputs/leave_one_subject_predictions.csv
- /content/patient_classifier_outputs/metrics_summary.json
- /content/patient_classifier_outputs/session_feature_matrix.csv
